In [8]:
from google.colab import drive
drive.mount('/content/drive')

!pip install lpips scikit-image pandas tqdm pillow torch torchvision

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim
import torch
import lpips
from torchvision import transforms
import pandas as pd


In [10]:
def find_render_dir(base_dir):
    """
    구조화된 경로 내에서 렌더링 결과 폴더를 자동으로 탐색합니다.
    1. base_dir/renders 가 존재하면 즉시 반환
    2. base_dir/**/test/ours_*/renders 패턴 검색
    """
    # 1. 직계 자식 'renders' 확인
    direct_path = os.path.join(base_dir, "renders")
    if os.path.exists(direct_path):
        return direct_path

    # 2. 특정 패턴 (test/ours_*/renders) 우선 검색
    search_pattern = os.path.join(base_dir, "**", "test", "ours_*", "renders")
    matches = glob.glob(search_pattern, recursive=True)
    if matches: return sorted(matches)[-1]

    # 3. 백업 로직: 이미지가 있고 'crop'이 아닌 첫 폴더
    for root, dirs, files in os.walk(base_dir):
        if 'crop' in root: continue
        if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files): return root
    return base_dir

def find_gt_dir(base_dir):
    """
    구조화된 경로 내에서 GT 폴더를 자동으로 탐색합니다.
    1. base_dir/gt 가 존재하면 즉시 반환
    2. base_dir/**/test/ours_*/gt 패턴 검색
    """
    # 1. 직계 자식 'gt' 확인
    direct_path = os.path.join(base_dir, "gt")
    if os.path.exists(direct_path):
        return direct_path

    # 2. 특정 패턴 (test/ours_*/gt) 우선 검색
    search_pattern = os.path.join(base_dir, "**", "test", "ours_*", "gt")
    matches = glob.glob(search_pattern, recursive=True)
    if matches: return sorted(matches)[-1]

    for root, dirs, files in os.walk(base_dir):
        if 'crop' in root: continue
        if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files): return root
    return base_dir


In [11]:
def center_crop(img, target_size=(1000, 640)):
    """Center crop image to target_size. Raises ValueError if image is smaller than target."""
    w, h = img.size
    tw, th = target_size
    if w < tw or h < th:
        raise ValueError(f"Image size ({w}x{h}) is smaller than target size ({tw}x{th})")
    left = (w - tw) // 2
    top = (h - th) // 2
    return img.crop((left, top, left + tw, top + th))


In [12]:
def load_image(path, for_lpips=False, apply_crop=True, target_size=(1000, 640), save_path=None):
    """Normalize image to [0,1] or convert to tensor for LPIPS. Optionally saves the (cropped) image."""
    img = Image.open(path).convert('RGB')
    if apply_crop: img = center_crop(img, target_size)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        img.save(save_path)
    if for_lpips:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
        return transform(img).unsqueeze(0)
    else:
        return np.array(img).astype(np.float32) / 255.0


In [13]:
def compute_all_metrics(gt_dir, pred_dir, lpips_net='alex', apply_crop=True, target_size=(1000, 640), save_crops=True):
    """
    Computes PSNR, SSIM, and LPIPS metrics between GT and prediction folders.
    Detailed output version.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    loss_fn = lpips.LPIPS(net=lpips_net).to(device).eval()

    gt_images = sorted([f for f in os.listdir(gt_dir) if f.endswith(('png', 'jpg', 'jpeg')) and not f.startswith('._')])
    pred_images = sorted([f for f in os.listdir(pred_dir) if f.endswith(('png', 'jpg', 'jpeg')) and not f.startswith('._')])

    if not gt_images or not pred_images:
        print(f"    [Warn] No images found! GT: {len(gt_images)}, Pred: {len(pred_images)}")
        return None

    if len(gt_images) != len(pred_images):
        print(f"    [Error] Count mismatch: GT({len(gt_images)}) vs Pred({len(pred_images)})")
        gt_set, pred_set = set(gt_images), set(pred_images)
        print(f"      - Only in GT: {list(gt_set - pred_set)[:3]}")
        print(f"      - Only in Pred: {list(pred_set - gt_set)[:3]}")
        raise ValueError("GT and prediction folders must contain the same number of images.")

    psnr_list, ssim_list, lpips_list = [], [], []
    pbar = tqdm(zip(gt_images, pred_images), total=len(gt_images), desc='    Processing', leave=False)
    
    for gt_img, pred_img in pbar:
        gt_path = os.path.join(gt_dir, gt_img)
        pred_path = os.path.join(pred_dir, pred_img)
        try:
            gt_np = load_image(gt_path, apply_crop=apply_crop, target_size=target_size, save_path=os.path.join(gt_dir, 'crop', gt_img) if save_crops else None)
            pred_np = load_image(pred_path, apply_crop=apply_crop, target_size=target_size, save_path=os.path.join(pred_dir, 'crop', pred_img) if save_crops else None)
            if gt_np.shape != pred_np.shape: continue
            psnr_val = compare_psnr(gt_np, pred_np, data_range=1.0)
            ssim_val = compare_ssim(gt_np, pred_np, data_range=1.0, channel_axis=2)
            gt_tensor = load_image(gt_path, for_lpips=True, apply_crop=apply_crop, target_size=target_size).to(device)
            pred_tensor = load_image(pred_path, for_lpips=True, apply_crop=apply_crop, target_size=target_size).to(device)
            with torch.no_grad():
                lpips_val = loss_fn(gt_tensor, pred_tensor).item()
            psnr_list.append(psnr_val); ssim_list.append(ssim_val); lpips_list.append(lpips_val)
            pbar.set_postfix({'PSNR': f'{psnr_val:.2f}'})
        except Exception as e: continue

    if not psnr_list: return None
    avg_psnr, avg_ssim, avg_lpips = np.mean(psnr_list), np.mean(ssim_list), np.mean(lpips_list)
    print(f"    => Results: PSNR: {avg_psnr:.3f} | SSIM: {avg_ssim:.3f} | LPIPS: {avg_lpips:.3f} ({len(psnr_list)} images)")
    return avg_psnr, avg_ssim, avg_lpips


In [14]:
DIR_BASE = '/content/drive/MyDrive/Research/26_Capstone2'
METHODS = ["LongSplat", "3DGS"]
CONDITIONS = ["original", "snow", "de_snow"]
SCENES = ["grass", "hydrant", "pillar", "road", "sky", "stair"]
APPLY_CROP = True  # Crop 활성화 여부 (True/False)

SCENE_LIST = [[os.path.join(DIR_BASE, f"{m}_{s}_{c}") for m in METHODS for c in CONDITIONS] for s in SCENES]
# v1 GT: LongSplat-Original-GT
GT_LIST = [[os.path.join(DIR_BASE, f"LongSplat_{s}_original") for m in METHODS for c in CONDITIONS] for s in SCENES]

print("=== 디렉토리 탐색 시작 (v1: GT=LongSplat-Original-GT) ===")
for i in range(len(SCENES)):
    for j in range(len(SCENE_LIST[i])):
        SCENE_LIST[i][j] = find_render_dir(SCENE_LIST[i][j])
        GT_LIST[i][j] = find_gt_dir(GT_LIST[i][j])

print("=== [v1] GT-Pred Pair Mapping Check ===")
for i, scene in enumerate(SCENES):
    print(f"[Scene: {scene}]")
    idx = 0
    for m in METHODS:
        for c in CONDITIONS:
            print(f"  {m} - {c}:")
            print(f"    GT:   {GT_LIST[i][idx]}")
            print(f"    PRED: {SCENE_LIST[i][idx]}")
            idx += 1


=== 디렉토리 탐색 시작 (v1: GT=LongSplat-Original-GT) ===
=== [v1] GT-Pred Pair Mapping Check ===
[Scene: grass]
  LongSplat - original:
    GT:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/gt
    PRED: /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/renders
  LongSplat - snow:
    GT:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/gt
    PRED: /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_snow/2026-04-17_01:26/test/ours_50000/renders
  LongSplat - de_snow:
    GT:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/gt
    PRED: /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_de_snow/2026-04-17_03:19/test/ours_50000/renders
  3DGS - original:
    GT:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/o

In [15]:
results = []
for i, (gt_group, pred_group) in enumerate(zip(GT_LIST, SCENE_LIST)):
    scene_name = SCENES[i]
    print('\n' + '='*50)
    print(f'   SCENE: {scene_name}')
    print('='*50)

    path_idx = 0
    for m_name in METHODS:
        for c_name in CONDITIONS:
            gt_dir = gt_group[path_idx]
            pred_dir = pred_group[path_idx]
            path_idx += 1
            
            print(f'\n   >>> Processing: {m_name} | {c_name}')
            print(f'      [GT]:   {gt_dir}')
            print(f'      [PRED]: {pred_dir}')

            if not os.path.exists(gt_dir):
                print(f'      [Warn] GT folder not found, skipping.')
                continue
            if not os.path.exists(pred_dir):
                print(f'      [Warn] PRED folder not found, skipping.')
                continue

            try:
                metrics = compute_all_metrics(gt_dir, pred_dir, apply_crop=APPLY_CROP, save_crops=True)
                if metrics:
                    psnr, ssim, lpips_val = metrics
                    results.append({
                        'Scene': scene_name,
                        'Method': m_name,
                        'Condition': c_name,
                        'PSNR': psnr,
                        'SSIM': ssim,
                        'LPIPS': lpips_val
                    })
            except Exception as e:
                print(f'      [Error] Analysis failed: {e}')
                continue

# [Summary] Save results and display table
if results:
    df = pd.DataFrame(results)
    csv_path = os.path.join(DIR_BASE, 'metrics_results_clean_raw.csv')
    df.to_csv(csv_path, index=False)
    print('\n' + '='*50)
    print('      FINAL EVALUATION SUMMARY')
    print('='*50)
    print(f'   >> Results saved to: {csv_path}')
    display(df)



   SCENE: grass

   >>> Processing: LongSplat | original
      [GT]:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/gt
      [PRED]: /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/renders
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


    => Results: PSNR: 22.236 | SSIM: 0.434 | LPIPS: 0.444 (25 images)

   >>> Processing: LongSplat | snow
      [GT]:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/gt
      [PRED]: /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_snow/2026-04-17_01:26/test/ours_50000/renders
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


    => Results: PSNR: 20.655 | SSIM: 0.377 | LPIPS: 0.544 (25 images)

   >>> Processing: LongSplat | de_snow
      [GT]:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_original/2026-04-16_23:40/test/ours_50000/gt
      [PRED]: /content/drive/MyDrive/Research/26_Capstone2/LongSplat/grass_de_snow/2026-04-17_03:19/test/ours_50000/renders
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


...

    => Results: PSNR: 27.197 | SSIM: 0.819 | LPIPS: 0.171 (25 images)

   >>> Processing: 3DGS | original
      [GT]:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/stair_original/2026-04-24_04:03/test/ours_50000/gt
      [PRED]: /content/drive/MyDrive/Research/26_Capstone2/3DGS/stair_original/2026-04-24_01:07/test/ours_30000/renders
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


    => Results: PSNR: 28.012 | SSIM: 0.833 | LPIPS: 0.133 (25 images)

   >>> Processing: 3DGS | snow
      [GT]:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/stair_original/2026-04-24_04:03/test/ours_50000/gt
      [PRED]: /content/drive/MyDrive/Research/26_Capstone2/3DGS/stair_snow/2026-04-24_01:46/test/ours_30000/renders
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


    => Results: PSNR: 24.708 | SSIM: 0.785 | LPIPS: 0.201 (25 images)

   >>> Processing: 3DGS | de_snow
      [GT]:   /content/drive/MyDrive/Research/26_Capstone2/LongSplat/stair_original/2026-04-24_04:03/test/ours_50000/gt
      [PRED]: /content/drive/MyDrive/Research/26_Capstone2/3DGS/stair_de_snow/2026-04-24_02:26/test/ours_30000/renders
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


    => Results: PSNR: 26.516 | SSIM: 0.798 | LPIPS: 0.172 (25 images)

      FINAL EVALUATION SUMMARY
   >> Results saved to: /content/drive/MyDrive/Research/26_Capstone2/metrics_results_clean_raw.csv


,Scene,Method,Condition,PSNR,SSIM,LPIPS
0,grass,LongSplat,original,22.235833,0.433649,0.444020
1,grass,LongSplat,snow,20.654972,0.376530,0.543611
2,grass,LongSplat,de_snow,21.640898,0.431078,0.466648
3,grass,3DGS,original,25.714391,0.702865,0.212573
4,grass,3DGS,snow,24.603863,0.684383,0.239476
5,grass,3DGS,de_snow,25.103221,0.684111,0.220104
6,hydrant,LongSplat,original,18.472370,0.386656,0.329351
7,hydrant,LongSplat,de_snow,9.899441,0.141007,0.815623
8,pillar,LongSplat,original,26.962899,0.761926,0.214155
9,pillar,LongSplat,snow,24.847509,0.757941,0.258986
